# Documentación de ClinicLog

ClinicLog es un sistema sencillo para registrar pacientes, tratamientos y seguimientos.

La idea del proyecto es que la información no esté separada o anotada de forma desordenada.
Por eso el sistema permite registrar pacientes, asignarles tratamientos, consultar su
información y llevar un historial básico.

## Importaciones

En esta parte se importan los módulos que se necesitan durante el programa.

`dataclass` se usa para crear las clases de pacientes, tratamientos y seguimientos.
`datetime` permite guardar la fecha y hora de cada seguimiento.
`List` y `Optional` ayudan a indicar los tipos de datos que se manejan.
Finalmente, `re` se usa para validar que los nombres solo contengan letras y espacios.

In [2]:
from dataclasses import dataclass
from datetime import datetime
from typing import List, Optional
import re

## Funciones de apoyo y validaciones

Estas funciones se usan en varias partes del sistema.

Permiten mostrar títulos en consola, leer textos, leer números enteros o decimales,
confirmar acciones y validar datos antes de guardarlos.

Las validaciones más importantes son:

- No permitir campos obligatorios vacíos.
- No permitir letras cuando se espera un número.
- Validar que la edad esté entre 0 y 120 años.
- Validar que el costo no sea negativo.
- Validar que nombre y apellido solo tengan letras y espacios.
- Confirmar antes de continuar con registros duplicados o eliminar un paciente.

In [3]:
def limpiar_pantalla():
    print("\n" * 2)


def mostrar_encabezado(titulo: str):
    linea = "=" * 40
    print(linea)
    print(titulo.center(40))
    print(linea)


def leer_cadena(mensaje: str, obligatorio: bool = True) -> str:
    while True:
        valor = input(mensaje).strip()
        if valor or not obligatorio:
            return valor
        print("Error: este campo es obligatorio.")


def leer_entero_rango(mensaje: str, minimo: int, maximo: int) -> int:
    while True:
        texto = input(mensaje).strip()
        try:
            numero = int(texto)
            if minimo <= numero <= maximo:
                return numero
            print(f"Error: ingrese un número entre {minimo} y {maximo}.")
        except ValueError:
            print("Error: ingrese un número entero válido.")


def leer_decimal_positivo(mensaje: str) -> float:
    while True:
        texto = input(mensaje).strip()
        try:
            numero = float(texto)
            if numero >= 0:
                return numero
            print("Error: el valor no puede ser negativo.")
        except ValueError:
            print("Error: ingrese un número decimal válido.")


def confirmar(mensaje: str) -> bool:
    while True:
        respuesta = input(f"{mensaje} (s/n): ").strip().lower()

        if respuesta in ("s", "si", "sí"):
            return True
        if respuesta in ("n", "no"):
            return False

        print("Error: responda con s o n.")


def generar_id() -> int:
    return int(datetime.now().timestamp() * 1000) % 1000000


def validar_nombre(nombre: str) -> bool:
    if len(nombre) < 2 or len(nombre) > 50:
        return False
    return bool(re.match(r"^[A-Za-zÁÉÍÓÚáéíóúÑñ\s]+$", nombre))

## Clases principales

Para representar la información del sistema se usan tres clases.

`Paciente` guarda los datos básicos de una persona.

`Tratamiento` guarda el nombre del tratamiento, dosis, frecuencia, costo y el paciente
al que pertenece.

`RegistroSeguimiento` guarda una observación relacionada con un paciente, por ejemplo,
una toma de medicamento, una visita o un control.

El identificador del paciente se guarda tanto en tratamientos como en seguimientos.
Esto permite saber a qué paciente corresponde cada dato.

In [4]:
@dataclass
class Paciente:
    id: int
    nombre: str
    apellido: str
    edad: int
    condicion_medica: Optional[str] = None

    def nombre_completo(self) -> str:
        return f"{self.nombre} {self.apellido}"


@dataclass
class Tratamiento:
    id: int
    id_paciente: int
    nombre: str
    dosis: str
    frecuencia_horas: int
    costo: float


@dataclass
class RegistroSeguimiento:
    id: int
    id_paciente: int
    id_tratamiento: int
    fecha: datetime
    observaciones: str

## Gestores y almacenamiento de datos

Los gestores se encargan de manejar las listas donde se guarda la información mientras
el programa está abierto.

- `GestorPacientes` registra, lista, verifica duplicados y elimina pacientes.
- `GestorTratamientos` registra tratamientos, busca los que pertenecen a un paciente
  y los elimina si ese paciente es eliminado.
- `GestorSeguimiento` guarda los registros de seguimiento, permite ver historiales
  y elimina los registros de un paciente cuando corresponde.

Cuando se elimina un paciente también se eliminan sus tratamientos y seguimientos.
Esto evita dejar información sin relación dentro del sistema.

In [5]:
class GestorPacientes:
    def __init__(self):
        self.pacientes: List[Paciente] = []

    def registrar(self, paciente: Paciente):
        self.pacientes.append(paciente)

    def listar(self) -> List[Paciente]:
        return sorted(self.pacientes, key=lambda p: (p.nombre, p.apellido))

    def existe(self, nombre: str, apellido: str, id_excluir: Optional[int] = None) -> bool:
        return any(
            p.id != id_excluir
            and p.nombre.lower() == nombre.lower()
            and p.apellido.lower() == apellido.lower()
            for p in self.pacientes
        )

    def eliminar(self, id_paciente: int):
        self.pacientes = [p for p in self.pacientes if p.id != id_paciente]


class GestorTratamientos:
    def __init__(self):
        self.tratamientos: List[Tratamiento] = []

    def registrar(self, tratamiento: Tratamiento):
        self.tratamientos.append(tratamiento)

    def por_paciente(self, id_paciente: int) -> List[Tratamiento]:
        return [t for t in self.tratamientos if t.id_paciente == id_paciente]

    def eliminar_por_paciente(self, id_paciente: int):
        self.tratamientos = [
            t for t in self.tratamientos
            if t.id_paciente != id_paciente
        ]


class GestorSeguimiento:
    def __init__(self):
        self.registros: List[RegistroSeguimiento] = []

    def agregar(self, registro: RegistroSeguimiento):
        self.registros.append(registro)

    def historial_paciente(self, id_paciente: int) -> List[RegistroSeguimiento]:
        return [
            r for r in self.registros
            if r.id_paciente == id_paciente
        ]

    def historial_tratamiento(
        self,
        id_paciente: int,
        id_tratamiento: int
    ) -> List[RegistroSeguimiento]:
        return [
            r for r in self.registros
            if r.id_paciente == id_paciente
            and r.id_tratamiento == id_tratamiento
        ]

    def eliminar_por_paciente(self, id_paciente: int):
        self.registros = [
            r for r in self.registros
            if r.id_paciente != id_paciente
        ]

## Datos de prueba y selección de paciente

Al iniciar el programa se crean dos pacientes, dos tratamientos y dos registros de seguimiento.
Esto permite probar las opciones de consultar pacientes y ver historiales desde el inicio.

También se creó una función para seleccionar pacientes. Esta función se reutiliza en
varias opciones, como registrar tratamiento, editar, eliminar y dar seguimiento.
Así no es necesario repetir el mismo código cada vez que se quiere seleccionar un paciente.

In [6]:
def inicializar_datos(
    gestor_pacientes: GestorPacientes,
    gestor_tratamientos: GestorTratamientos,
    gestor_seguimiento: GestorSeguimiento
) -> None:
    p1 = Paciente(
        generar_id(),
        "Germán",
        "Pérez",
        35,
        "Hipertensión"
    )

    p2 = Paciente(
        generar_id(),
        "Ana",
        "Gómez",
        28,
        "Diabetes tipo 2"
    )

    gestor_pacientes.registrar(p1)
    gestor_pacientes.registrar(p2)

    t1 = Tratamiento(
        generar_id(),
        p1.id,
        "Losartán",
        "50mg",
        24,
        150.0
    )

    t2 = Tratamiento(
        generar_id(),
        p2.id,
        "Metformina",
        "850mg",
        12,
        80.0
    )

    gestor_tratamientos.registrar(t1)
    gestor_tratamientos.registrar(t2)

    ahora = datetime.now()

    gestor_seguimiento.agregar(
        RegistroSeguimiento(
            generar_id(),
            p1.id,
            t1.id,
            ahora,
            "Toma de prueba"
        )
    )

    gestor_seguimiento.agregar(
        RegistroSeguimiento(
            generar_id(),
            p2.id,
            t2.id,
            ahora,
            "Control inicial"
        )
    )


def seleccionar_paciente(
    gestor_pacientes: GestorPacientes
) -> Optional[Paciente]:
    pacientes = gestor_pacientes.listar()

    if not pacientes:
        print("No hay pacientes registrados.")
        input("Presione Enter para continuar...")
        return None

    for i, paciente in enumerate(pacientes, 1):
        print(f"{i}. {paciente.nombre_completo()} ({paciente.edad} años)")

    indice = leer_entero_rango(
        "Seleccione paciente (número): ",
        1,
        len(pacientes)
    )

    return pacientes[indice - 1]

## Registro de pacientes y tratamientos

Estas dos funciones sirven para crear nuevos registros.

La primera registra pacientes. Pide nombre, apellido, edad y condición médica.
Antes de guardar, valida los nombres, la edad y revisa si ya existe alguien con el mismo
nombre y apellido.

La segunda registra tratamientos. Primero se selecciona un paciente y luego se ingresan
el nombre del tratamiento, la dosis, la frecuencia y el costo.

In [7]:
def registrar_paciente_menu(
    gestor_pacientes: GestorPacientes
) -> None:
    mostrar_encabezado("REGISTRAR PACIENTE")

    while True:
        nombre = leer_cadena("Nombre: ")

        if validar_nombre(nombre):
            break

        print("Error: el nombre solo puede contener letras (2-50 caracteres).")

    while True:
        apellido = leer_cadena("Apellido: ")

        if validar_nombre(apellido):
            break

        print("Error: el apellido solo puede contener letras (2-50 caracteres).")

    if gestor_pacientes.existe(nombre, apellido):
        print("Advertencia: ya existe un paciente con este nombre y apellido.")

        if not confirmar("¿Desea continuar y registrar otro?"):
            print("Operación cancelada.")
            input("Presione Enter para continuar...")
            return

    edad = leer_entero_rango("Edad: ", 0, 120)

    condicion = leer_cadena(
        "Condición médica (opcional, Enter para saltar): ",
        obligatorio=False
    )

    if condicion == "":
        condicion = None

    paciente = Paciente(
        generar_id(),
        nombre,
        apellido,
        edad,
        condicion
    )

    gestor_pacientes.registrar(paciente)

    print(f"Paciente {paciente.nombre_completo()} registrado.")
    input("Presione Enter para continuar...")


def registrar_tratamiento_menu(
    gestor_pacientes: GestorPacientes,
    gestor_tratamientos: GestorTratamientos
) -> None:
    mostrar_encabezado("REGISTRAR TRATAMIENTO")

    paciente = seleccionar_paciente(gestor_pacientes)

    if paciente is None:
        return

    nombre = leer_cadena("Nombre del tratamiento: ")
    dosis = leer_cadena("Dosis: ")
    frecuencia = leer_entero_rango(
        "Frecuencia (horas entre tomas): ",
        1,
        365
    )
    costo = leer_decimal_positivo("Costo: ")

    tratamiento = Tratamiento(
        generar_id(),
        paciente.id,
        nombre,
        dosis,
        frecuencia,
        costo
    )

    gestor_tratamientos.registrar(tratamiento)

    print(
        f"Tratamiento '{nombre}' registrado para "
        f"{paciente.nombre_completo()}."
    )

    input("Presione Enter para continuar...")

## Gestión de pacientes: consultar, editar y eliminar

En esta parte se completa el CRUD de pacientes.

Primero se pueden ver los pacientes y los tratamientos que tienen registrados.

También se puede editar un paciente. Al editar, se muestran los datos actuales y se puede
presionar Enter si se quiere conservar un valor. Se validan nuevamente nombre, apellido
y edad antes de guardar los cambios.

Por último, se puede eliminar un paciente. Antes de eliminar se pide confirmación. Además,
se borran sus tratamientos y registros de seguimiento para no dejar datos relacionados con
un paciente que ya no existe.

In [8]:
def ver_pacientes(
    gestor_pacientes: GestorPacientes,
    gestor_tratamientos: GestorTratamientos
) -> None:
    mostrar_encabezado("LISTA DE PACIENTES")

    pacientes = gestor_pacientes.listar()

    if not pacientes:
        print("No hay pacientes registrados.")
        input("Presione Enter para continuar...")
        return

    for i, paciente in enumerate(pacientes, 1):
        print(f"{i}. {paciente.nombre_completo()} - {paciente.edad} años")

        if paciente.condicion_medica:
            print(f"   Condición: {paciente.condicion_medica}")

        tratamientos = gestor_tratamientos.por_paciente(paciente.id)

        if tratamientos:
            print("   Tratamientos:")

            for tratamiento in tratamientos:
                print(
                    f"      - {tratamiento.nombre} "
                    f"({tratamiento.dosis}) cada "
                    f"{tratamiento.frecuencia_horas}h, "
                    f"costo {tratamiento.costo:.2f}"
                )
        else:
            print("   Sin tratamientos")

        print()

    input("Presione Enter para continuar...")


def editar_paciente_menu(
    gestor_pacientes: GestorPacientes
) -> None:
    mostrar_encabezado("EDITAR PACIENTE")

    paciente = seleccionar_paciente(gestor_pacientes)

    if paciente is None:
        return

    print("Presione Enter para conservar el valor actual.")

    while True:
        nuevo_nombre = input(
            f"Nombre [{paciente.nombre}]: "
        ).strip()

        if nuevo_nombre == "":
            nuevo_nombre = paciente.nombre
            break

        if validar_nombre(nuevo_nombre):
            break

        print("Error: el nombre solo puede contener letras (2-50 caracteres).")

    while True:
        nuevo_apellido = input(
            f"Apellido [{paciente.apellido}]: "
        ).strip()

        if nuevo_apellido == "":
            nuevo_apellido = paciente.apellido
            break

        if validar_nombre(nuevo_apellido):
            break

        print("Error: el apellido solo puede contener letras (2-50 caracteres).")

    if gestor_pacientes.existe(
        nuevo_nombre,
        nuevo_apellido,
        paciente.id
    ):
        print("Error: ya existe otro paciente con ese nombre y apellido.")
        input("Presione Enter para continuar...")
        return

    while True:
        texto_edad = input(
            f"Edad [{paciente.edad}]: "
        ).strip()

        if texto_edad == "":
            nueva_edad = paciente.edad
            break

        try:
            nueva_edad = int(texto_edad)

            if 0 <= nueva_edad <= 120:
                break

            print("Error: ingrese una edad entre 0 y 120.")

        except ValueError:
            print("Error: ingrese un número entero válido.")

    nueva_condicion = input(
        f"Condición médica [{paciente.condicion_medica or 'Sin registrar'}]: "
    ).strip()

    if nueva_condicion == "":
        nueva_condicion = paciente.condicion_medica

    paciente.nombre = nuevo_nombre
    paciente.apellido = nuevo_apellido
    paciente.edad = nueva_edad
    paciente.condicion_medica = nueva_condicion

    print("Paciente actualizado correctamente.")
    input("Presione Enter para continuar...")


def eliminar_paciente_menu(
    gestor_pacientes: GestorPacientes,
    gestor_tratamientos: GestorTratamientos,
    gestor_seguimiento: GestorSeguimiento
) -> None:
    mostrar_encabezado("ELIMINAR PACIENTE")

    paciente = seleccionar_paciente(gestor_pacientes)

    if paciente is None:
        return

    print(
        f"Se eliminará a {paciente.nombre_completo()} y también "
        "sus tratamientos y registros de seguimiento."
    )

    if not confirmar("¿Desea continuar?"):
        print("Operación cancelada.")
        input("Presione Enter para continuar...")
        return

    gestor_tratamientos.eliminar_por_paciente(paciente.id)
    gestor_seguimiento.eliminar_por_paciente(paciente.id)
    gestor_pacientes.eliminar(paciente.id)

    print("Paciente eliminado correctamente.")
    input("Presione Enter para continuar...")


def gestionar_pacientes_menu(
    gestor_pacientes: GestorPacientes,
    gestor_tratamientos: GestorTratamientos,
    gestor_seguimiento: GestorSeguimiento
) -> None:
    while True:
        mostrar_encabezado("GESTIONAR PACIENTES")
        print("1. Ver pacientes")
        print("2. Editar paciente")
        print("3. Eliminar paciente")
        print("4. Volver")

        opcion = leer_entero_rango(
            "Seleccione una opción (1-4): ",
            1,
            4
        )

        if opcion == 1:
            ver_pacientes(
                gestor_pacientes,
                gestor_tratamientos
            )

        elif opcion == 2:
            editar_paciente_menu(gestor_pacientes)

        elif opcion == 3:
            eliminar_paciente_menu(
                gestor_pacientes,
                gestor_tratamientos,
                gestor_seguimiento
            )

        elif opcion == 4:
            break

## Seguimiento de pacientes

Esta parte permite trabajar con el historial de un paciente.

Después de seleccionar un paciente, se puede ver todo su historial, consultar solo los
registros de un tratamiento o agregar una nueva toma, visita u observación.

Cuando se agrega un seguimiento, el sistema guarda la fecha y hora actual junto con lo
que se escribió en observaciones.

In [9]:
def seguimiento_paciente_menu(
    gestor_pacientes: GestorPacientes,
    gestor_tratamientos: GestorTratamientos,
    gestor_seguimiento: GestorSeguimiento
) -> None:
    mostrar_encabezado("SEGUIMIENTO DE PACIENTE")

    paciente = seleccionar_paciente(gestor_pacientes)

    if paciente is None:
        return

    tratamientos = gestor_tratamientos.por_paciente(paciente.id)

    while True:
        mostrar_encabezado(
            f"SEGUIMIENTO: {paciente.nombre_completo()}"
        )

        print("1. Ver historial completo")
        print("2. Ver historial de un tratamiento")
        print("3. Registrar nueva toma/visita")
        print("4. Volver")

        opcion = leer_entero_rango("Opción: ", 1, 4)

        if opcion == 1:
            historial = gestor_seguimiento.historial_paciente(
                paciente.id
            )

            if not historial:
                print("No hay registros para este paciente.")
            else:
                for registro in historial:
                    print(
                        f"- {registro.fecha.strftime('%d/%m/%Y %H:%M')} | "
                        f"{registro.observaciones}"
                    )

            input("Presione Enter para continuar...")

        elif opcion == 2:
            if not tratamientos:
                print("Este paciente no tiene tratamientos.")
                input("Presione Enter para continuar...")
                continue

            for i, tratamiento in enumerate(tratamientos, 1):
                print(f"{i}. {tratamiento.nombre}")

            seleccion = leer_entero_rango(
                "Seleccione tratamiento: ",
                1,
                len(tratamientos)
            )

            tratamiento = tratamientos[seleccion - 1]

            historial = gestor_seguimiento.historial_tratamiento(
                paciente.id,
                tratamiento.id
            )

            if not historial:
                print("No hay registros para este tratamiento.")
            else:
                for registro in historial:
                    print(
                        f"- {registro.fecha.strftime('%d/%m/%Y %H:%M')} | "
                        f"{registro.observaciones}"
                    )

            input("Presione Enter para continuar...")

        elif opcion == 3:
            if tratamientos:
                print("Tratamientos del paciente:")

                for i, tratamiento in enumerate(tratamientos, 1):
                    print(f"{i}. {tratamiento.nombre}")

                seleccion = leer_entero_rango(
                    "Seleccione tratamiento: ",
                    1,
                    len(tratamientos)
                )

                tratamiento = tratamientos[seleccion - 1]
                id_tratamiento = tratamiento.id

            else:
                id_tratamiento = 0

            observaciones = leer_cadena("Observaciones: ")

            registro = RegistroSeguimiento(
                generar_id(),
                paciente.id,
                id_tratamiento,
                datetime.now(),
                observaciones
            )

            gestor_seguimiento.agregar(registro)

            print("Registro agregado.")
            input("Presione Enter para continuar...")

        elif opcion == 4:
            break

## Menú principal y ejecución

El menú principal mantiene las cinco opciones solicitadas.

La opción tres contiene un submenú para gestionar pacientes. Ahí se pueden consultar,
editar o eliminar, sin agregar más opciones al menú principal.

La función `main` crea las listas, carga los datos de prueba y mantiene el programa
funcionando hasta que el usuario selecciona salir.

In [10]:
def menu_principal():
    mostrar_encabezado("CLINICLOG - MENÚ PRINCIPAL")
    print("1. Registrar paciente")
    print("2. Registrar tratamiento")
    print("3. Gestionar pacientes")
    print("4. Seguimiento de paciente")
    print("5. Salir")


def main():
    gestor_pacientes = GestorPacientes()
    gestor_tratamientos = GestorTratamientos()
    gestor_seguimiento = GestorSeguimiento()

    inicializar_datos(
        gestor_pacientes,
        gestor_tratamientos,
        gestor_seguimiento
    )

    while True:
        limpiar_pantalla()
        menu_principal()

        opcion = leer_entero_rango(
            "Seleccione una opción (1-5): ",
            1,
            5
        )

        if opcion == 1:
            registrar_paciente_menu(gestor_pacientes)

        elif opcion == 2:
            registrar_tratamiento_menu(
                gestor_pacientes,
                gestor_tratamientos
            )

        elif opcion == 3:
            gestionar_pacientes_menu(
                gestor_pacientes,
                gestor_tratamientos,
                gestor_seguimiento
            )

        elif opcion == 4:
            seguimiento_paciente_menu(
                gestor_pacientes,
                gestor_tratamientos,
                gestor_seguimiento
            )

        elif opcion == 5:
            mostrar_encabezado("GRACIAS POR USAR CLINICLOG")
            break